In [106]:
import re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from gensim.models import Word2Vec
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [107]:
dataframe = pd.read_csv("dataset/07-yelp-dataset.txt", sep="\t", names=["review", "label"])
X_train, X_test, y_train, y_test = train_test_split(dataframe["review"], dataframe["label"], test_size=0.20, random_state=42)
print(len(X_train), len(X_test))

800 200


In [108]:
def tokenize(text):
    return re.sub(r"[^a-z\s]", "", text.lower()).split()

train_tokens = [tokenize(review) for review in X_train]
test_tokens = [tokenize(review) for review in X_test]
train_tokens[:3]

[['the', 'worst', 'was', 'the', 'salmon', 'sashimi'],
 ['an',
  'excellent',
  'new',
  'restaurant',
  'by',
  'an',
  'experienced',
  'frenchman'],
 ['went', 'for', 'lunch', 'service', 'was', 'slow']]

In [109]:
w2v_model = Word2Vec(train_tokens, vector_size=200, window=5, min_count=1, sg=1)
print(len(w2v_model.wv))

1801


In [110]:
def review_vector(tokens):
    vectors = [w2v_model.wv[word] for word in tokens if word in w2v_model.wv]
    return np.mean(vectors, axis=0) if vectors else np.zeros(200)

X_train_tensor = torch.FloatTensor(np.array([review_vector(t) for t in train_tokens]))
X_test_tensor = torch.FloatTensor(np.array([review_vector(t) for t in test_tokens]))
y_train_tensor = torch.FloatTensor(y_train.tolist()).reshape(-1, 1)
print(X_train_tensor.shape, X_test_tensor.shape)

torch.Size([800, 200]) torch.Size([200, 200])


In [111]:
model = nn.Sequential(
    nn.Linear(200, 128),
    nn.ReLU(),
    nn.Linear(128, 64),
    nn.ReLU(),
    nn.Linear(64, 32),
    nn.ReLU(),
    nn.Linear(32, 1)
)
print(model)

Sequential(
  (0): Linear(in_features=200, out_features=128, bias=True)
  (1): ReLU()
  (2): Linear(in_features=128, out_features=64, bias=True)
  (3): ReLU()
  (4): Linear(in_features=64, out_features=32, bias=True)
  (5): ReLU()
  (6): Linear(in_features=32, out_features=1, bias=True)
)


In [112]:
loss_function = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(15):
    model.train()
    outputs = model(X_train_tensor)
    loss = loss_function(outputs, y_train_tensor)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    print(f"Epoch {epoch + 1}/15 - Loss: {loss.item():.4f}")

Epoch 1/15 - Loss: 0.6931
Epoch 2/15 - Loss: 0.6931
Epoch 3/15 - Loss: 0.6930
Epoch 4/15 - Loss: 0.6930
Epoch 5/15 - Loss: 0.6930
Epoch 6/15 - Loss: 0.6930
Epoch 7/15 - Loss: 0.6930
Epoch 8/15 - Loss: 0.6929
Epoch 9/15 - Loss: 0.6929
Epoch 10/15 - Loss: 0.6929
Epoch 11/15 - Loss: 0.6929
Epoch 12/15 - Loss: 0.6928
Epoch 13/15 - Loss: 0.6928
Epoch 14/15 - Loss: 0.6928
Epoch 15/15 - Loss: 0.6927


In [113]:
model.eval()
with torch.no_grad():
    predictions = (torch.sigmoid(model(X_test_tensor)) >= 0.5).int().numpy().flatten()

print("Test Accuracy:", accuracy_score(y_test, predictions))

Test Accuracy: 0.5
